# 05 — Análisis de Sentimiento
**Goal:** Detectar la polaridad (positivo / negativo / neutro) de comentarios de clientes.

Dos enfoques:
```
1. Basado en léxico  → diccionario de palabras con puntaje de sentimiento
                       rápido, sin datos etiquetados, menos preciso

2. Basado en ML      → TF-IDF + clasificador entrenado sobre ejemplos etiquetados
                       más preciso, requiere datos etiquetados
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','algo','ante','como','con','de','del','desde','el','ella','en',
    'entre','es','esta','este','esto','fue','la','las','le','lo','los','me',
    'mi','muy','no','nos','o','para','pero','por','que','se','si','sin',
    'su','sus','también','te','todo','un','una','y','ya','yo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
templates_pos = [
    'El proceso de solicitud fue muy rápido y eficiente',
    'Excelente servicio, aprobaron mi tarjeta rápidamente',
    'La app funciona perfecto, muy fácil de usar',
    'Increíbles beneficios, recomiendo la tarjeta al 100',
    'Atención al cliente excepcional, resolvieron todo rápido',
    'Proceso de activación muy sencillo y sin problemas',
    'Muy satisfecho con el servicio y la experiencia',
    'La tarjeta tiene beneficios increíbles y el proceso fue fácil',
]
templates_neg = [
    'La app está muy lenta, tardé mucho en activar la tarjeta',
    'No me llegó el OTP, tuve que llamar varias veces',
    'La carga de documentos falla constantemente',
    'Llevo días esperando la evaluación crediticia sin respuesta',
    'El call center no responde, pésimo servicio al cliente',
    'El proceso de aprobación es muy lento e ineficiente',
    'Muy mal servicio, nadie resuelve mis problemas',
    'La app falla todo el tiempo y el soporte es terrible',
]

rng = np.random.default_rng(42)
pos = [templates_pos[rng.integers(8)] for _ in range(400)]
neg = [templates_neg[rng.integers(8)] for _ in range(400)]
comments   = pos + neg
sentiments = [1]*400 + [0]*400

df = pd.DataFrame({'comment': comments, 'sentiment': sentiments})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Dataset: {len(df)} comentarios | 50% positivos')

## 1. Enfoque léxico — diccionario de sentimiento

In [ ]:
# Léxico de sentimiento en español (fintech / experiencia de usuario)
LEXICON_POS = {
    'excelente': 3, 'increíble': 3, 'excepcional': 3, 'perfecto': 3,
    'rápido': 2, 'rápidamente': 2, 'eficiente': 2, 'fácil': 2,
    'sencillo': 2, 'recomiendo': 2, 'satisfecho': 2, 'bueno': 1,
    'bien': 1, 'beneficios': 1, 'aprobaron': 1, 'resolvieron': 1,
}

LEXICON_NEG = {
    'lenta': -2, 'lento': -2, 'falla': -2, 'pésimo': -3, 'terrible': -3,
    'ineficiente': -2, 'tardé': -1, 'espera': -1, 'esperando': -1,
    'problema': -2, 'problemas': -2, 'mal': -2, 'no': -1, 'nadie': -2,
}

NEGATORS = {'no', 'nunca', 'jamás', 'sin', 'ni'}

def lexicon_score(text, flip_on_negation=True):
    tokens = preprocess(text).split()
    score = 0
    for i, tok in enumerate(tokens):
        word_score = LEXICON_POS.get(tok, 0) + LEXICON_NEG.get(tok, 0)
        if flip_on_negation and i > 0 and tokens[i-1] in NEGATORS:
            word_score *= -1
        score += word_score
    return score

# Aplicar al dataset
df['lex_score'] = df['comment'].apply(lexicon_score)
df['lex_pred']  = (df['lex_score'] > 0).astype(int)

from sklearn.metrics import accuracy_score, f1_score
print(f'Léxico — Accuracy: {accuracy_score(df.sentiment, df.lex_pred):.3f}')
print(f'Léxico — F1:       {f1_score(df.sentiment, df.lex_pred):.3f}')

In [ ]:
# Distribución de scores por clase
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df[df.sentiment==1]['lex_score'], bins=15, alpha=0.7,
        color='#06d6a0', label='Positivo (real)')
ax.hist(df[df.sentiment==0]['lex_score'], bins=15, alpha=0.7,
        color='#f72585', label='Negativo (real)')
ax.axvline(0, color='black', linewidth=1.5, linestyle='--', label='Umbral')
ax.set_xlabel('Score léxico')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de scores léxicos por clase real')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Enfoque ML — TF-IDF + Clasificador

In [ ]:
X = df['comment']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = [
    ('Logistic Regression',    LogisticRegression(max_iter=1000, C=1.0)),
    ('Complement Naive Bayes', ComplementNB()),   # especializado para texto
]

print(f'{"Modelo":<28} {"CV F1":>8}  {"Test F1":>8}  {"Test Acc":>9}')
print('-' * 60)

pipes = {}
for name, clf in models:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                                  ngram_range=(1, 2), sublinear_tf=True, min_df=2)),
        ('clf', clf),
    ])
    cv_f1  = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1').mean()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f'{name:<28} {cv_f1:8.4f}  '
          f'{f1_score(y_test, y_pred):8.4f}  '
          f'{accuracy_score(y_test, y_pred):9.4f}')
    pipes[name] = pipe

In [ ]:
# Reporte detallado del mejor modelo
best = pipes['Logistic Regression']
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Negativo', 'Positivo']))

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Negativo', 'Positivo'],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — TF-IDF + Logistic Regression')
plt.tight_layout()
plt.show()

## 3. Palabras más predictivas por clase

In [ ]:
coefs = best.named_steps['clf'].coef_[0]
fn    = best.named_steps['tfidf'].get_feature_names_out()

top_pos = np.argsort(coefs)[::-1][:15]
top_neg = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(fn[top_pos][::-1], coefs[top_pos][::-1], color='#06d6a0')
axes[0].set_title('Top features → Positivo')
axes[0].set_xlabel('Coeficiente')

axes[1].barh(fn[top_neg], np.abs(coefs[top_neg]), color='#f72585')
axes[1].set_title('Top features → Negativo')
axes[1].set_xlabel('|Coeficiente|')

plt.suptitle('Features más discriminativas — TF-IDF + LR', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Predicción en nuevos comentarios

In [ ]:
nuevos = [
    'Increíble, la tarjeta llegó en 2 días',
    'Llevan 5 días y nadie resuelve mi caso',
    'La app mejoró bastante, antes era lentísima',
    'No entiendo por qué rechazaron mi solicitud',
    'Todo perfecto, proceso rápido y sin problemas',
]

probs = best.predict_proba(nuevos)[:, 1]  # P(positivo)
preds = best.predict(nuevos)

print(f'{"Comentario":<50} {"P(pos)":>7}  {"Predicción":>10}')
print('-' * 75)
for text, prob, pred in zip(nuevos, probs, preds):
    label = '😊 Positivo' if pred == 1 else '😤 Negativo'
    print(f'{text:<50} {prob:7.3f}  {label}')

## Resumen
| Enfoque | Pros | Contras |
|---|---|---|
| Léxico | Sin entrenamiento, interpretable | Necesita léxico ad-hoc, baja precisión |
| TF-IDF + LR | Alta precisión, rápido | Necesita datos etiquetados |
| Complement NB | Muy rápido, bueno en desbalance | Supone independencia de features |

| API | Para qué |
|---|---|
| `LogisticRegression` | Clasificador default para texto |
| `ComplementNB` | Alternativa a MultinomialNB, mejor en clases desbalanceadas |
| `pipe.predict_proba()` | Probabilidades — útil para umbrales personalizados |
| `clf.coef_` | Interpretabilidad: qué palabras mueven el score |

**Siguiente:** `06_topic_modeling.ipynb` — descubrir temas automáticamente con LDA.